### Open TO-DOs:

1. Add PII to the essays with Faker
2. Fine-tune GEMMA on competition data
3. Diversify prompts

In [ ]:
!pip install -q -U --upgrade pip
!pip install -q tensorflow-text
!pip install -q -U keras
!pip install -q -U tensorflow
!pip install -q -U --upgrade tensorflow-hub
!pip install -q -U --upgrade tensorflow-cpu 
!pip install -q -U --upgrade keras-nlp 
!pip install -q -U --upgrade keras>=3

In [ ]:
import os
keras_backend = "jax"
allocation_fraction = 0.9
tf_min_log_level = '3'
os.environ["KERAS_BACKEND"] = keras_backend
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = str(allocation_fraction)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = str(tf_min_log_level)

In [ ]:
import jax
import keras_nlp
import keras
import time
import copy

import string
import random
import numpy as np
import pandas as pd
letters = [letter for letter in string.ascii_letters]

print(jax.devices())
print(keras.backend.backend())

In [ ]:
SEED = 101
# Seed the same seed to all 
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)

In [ ]:
# Note we use jax.devices() instead of keras.distribution.list_devices()
def initialize_device_mesh(
    shape: tuple[int, int] = (1, 8), 
    batch_axis_name: str = "batch",
    model_axis_name: str = "model"
) -> keras.distribution.DeviceMesh:
    """Initializes and returns a DeviceMesh for distributing computation across devices.
    
    To load the model with the weights and tensors distributed across TPUs, we first create a new DeviceMesh. 
        - DeviceMesh represents a collection of hardware devices configured for distributed computation.
        - DeviceMesh was introduced in Keras 3 as part of the unified distribution API.
    
    The distribution API enables data and model parallelism.
        - This allows for efficient scaling of deep learning models on multiple accelerators and hosts. 
        - The API leverages the underlying framework (e.g. JAX) to distribute the program and tensors according to the sharding directives.
            - This is done through a procedure called single program, multiple data (SPMD) expansion. 
            - Check out more details in the new Keras 3 distribution API guide.
                --> https://keras.io/guides/distribution/
    
    Args:
        shape: A tuple specifying the shape of the overall `DeviceMesh` 
            - `(8,)` for a data parallel only distribution,
            - `(4, 2)` for a model+data parallel distribution.
        batch_axis_name: A string indicating the axis name for the batch axis for DeviceMesh
        model_axis_name: The logical name of the model axis for the `DeviceMesh`

    Returns:
        A configured DeviceMesh instance. 
            - Defaults to (1, 8) shape so that the weights are sharded across all 8 TPUs (v3-8).
        NOTE: This API is aligned with `jax.sharding.Mesh` and `tf.dtensor.Mesh`
            - i.e. It represents the computation devices in the global context.
    """
    return keras.distribution.DeviceMesh(
        shape=shape, 
        axis_names=[batch_axis_name, model_axis_name], 
        devices=jax.devices()
    )


def configure_layout_map(
    device_mesh: keras.distribution.DeviceMesh,
    model_axis_name: str = "model"
) -> keras.distribution.LayoutMap:
    """Configures and returns a LayoutMap for model weight distribution.
    
    LayoutMap from the distribution API specifies how the weights and tensors should be sharded or replicated, using the string keys. 
        - For example: 'token_embedding/embeddings' below, which are treated like regex to match tensor paths. 
        - Matched tensors are sharded with model dimensions (8 TPUs); others will be fully replicated.

    Args:
        device_mesh: The `DeviceMesh` that is used to populate the `TensorLayout.device_mesh`
        axis_name: The logical name of the model axis for the `DeviceMesh`

    Returns:
        A LayoutMap instance with predefined sharding configurations.
    """

    # A dict-like object that maps string to `TensorLayout` instances.
    layout_map = keras.distribution.LayoutMap(device_mesh)

    # Weights that match 'token_embedding/embeddings' will be sharded on 8 TPUs
    layout_map["token_embedding/embeddings"] = (None, model_axis_name)
    
    # Regex to match against the query, key and value matrices in the decoder attention layers
    layout_map["decoder_block.*attention.*(query|key|value).*kernel"] = (None, model_axis_name, None)
    
    # etc.
    layout_map["decoder_block.*attention_output.*kernel"] = (None, None, model_axis_name)
    layout_map["decoder_block.*ffw_gating.*kernel"] = (model_axis_name, None)
    layout_map["decoder_block.*ffw_linear.*kernel"] = (None, model_axis_name)
    
    return layout_map


def update_distribution_strategy(
    device_mesh: keras.distribution.DeviceMesh, 
    layout_map: keras.distribution.LayoutMap
) -> None:
    """Loads a distributed Gemma model based on the provided device mesh and layout map.
    
    ModelParallel allows you to shard model weights or activation tensors across all devcies on the DeviceMesh.
    In this case, some of the Gemma 7B model weights are sharded across 8 TPU chips according the layout_map.
    
    Args:
        model_name: The name of the Gemma model to load.
        device_mesh: A DeviceMesh instance for model distribution.
        layout_map: A LayoutMap instance defining how to distribute model weights.

    Returns:
        None; The keras backend is updated with the appropriate distribution strategy
    """
    
    # Shard across devices in the mesh and update distribution strategy accordingly
    model_parallel = keras.distribution.ModelParallel(device_mesh, layout_map, batch_dim_name="batch")
    keras.distribution.set_distribution(model_parallel)
    
    
def get_distributed_gemma(model_name: str) -> keras_nlp.models.GemmaCausalLM:
    """Obtain the TPU compatible Gemma model of your choice.
    
    Args:
        model_name: The name of the Gemma model to load. One of:
            - 'gemma_2b_en'
            - 'gemma_7b_en'
            - 'gemma_instruct_2b_en'
            - 'gemma_instruct_7b_en'
        
    Returns:
        A loaded Gemma model instance configured for distributed computation.
    """
    # Return the model 
    return keras_nlp.models.GemmaCausalLM.from_preset(model_name)


def do_gemma_prep(return_device_mesh=True, return_layout_map=True):
    """Does the necessary steps so that we can instantiate a model properly
    
    Args:
        return_*: Whether to return the respective object as part of a dictionary
            - The key is the name (*) and the value is the object itself
    """
    device_mesh = initialize_device_mesh()
    layout_map = configure_layout_map(device_mesh)
    update_distribution_strategy(device_mesh, layout_map)
    
    return_map = {}
    if not (return_device_mesh or return_layout_map):
        pass
    else:
        if return_device_mesh:
            return_map["device_mesh"] = device_mesh
        if return_layout_map:
            return_map["layout_map"] = layout_map
    return return_map

### Before you use this:

- go to https://www.kaggle.com/models/keras/gemma/frameworks/Keras/variations/gemma_7b_en/versions/1
- click on "View license consent"
- Consent

In [ ]:
%%time
setup_objects = do_gemma_prep()
model = get_distributed_gemma("gemma_instruct_7b_en")
# This is necessary to generate more variating essays (else same input = same output)
model.compile(sampler=keras_nlp.samplers.TopKSampler(k=5, temperature=0.7))

# Read in Generated PII

In [ ]:
pii_df = pd.read_csv("/kaggle/input/pii-detect-mistral7b-pii-generation/pii_mistral.csv")

pii_df.shape

In [ ]:
pii_df.pii_type.value_counts()

In [ ]:
pii_data = {pii_type: pii_df[pii_df['pii_type'] == pii_type]['pii_id'].tolist() for pii_type in pii_df['pii_type'].unique()}

pii_data.keys()

In [ ]:
pii_types_distribution = {
                                        'NAME_STUDENT': 0.7,
                                        'URL_PERSONAL': 0.1,
                                        'ID_NUM': 0.05,
                                        'EMAIL': 0.05,
                                        'PHONE_NUM': 0.05,
                                        'STREET_ADDRESS': 0.05
                                    }

# Generate Essay function

In [ ]:
def generate_essay(length, num_pii=0, pii_df=None):
    topic = np.random.choice(['Learning Launch', 'Visualization', 'Storytelling', 'Visual Thinking', 'Mind mapping', 'Design thinking'])
    
    if pii_df is not None: 
    
        # Determine PII types to include in the essay
        pii_selection = np.random.choice(list(pii_types_distribution.keys()), 
                                         size=num_pii, 
                                         replace=True, 
                                         p=list(pii_types_distribution.values()))

        # Sample PIIs from the pii_df based on pii_selection
        pii_details = ""
        for pii_type in pii_selection:
            pii_sample = pii_df[pii_df['pii_type'] == pii_type]['pii_id'].sample(1).values[0]
            pii_details += f"{pii_type}: {pii_sample}\n"

        prompt_template = f"""
        You are a student enrolled in a massively open online course (MOOC) and are tasked with writing an essay on "{topic}" to apply the course material to address a real-world problem. As part of your essay, you decide to include a personal story or example that directly involves you or someone you know, which includes the following personal information (PII) to enrich the narrative:

        {pii_details}

        In your essay, make sure to creatively integrate this personal information in a way that enhances your argument or story. You might use the STREET_ADDRESS to set the scene, the URL_PERSONAL as a reference to an online portfolio or project, and the NAME_STUDENT to personalize the story, for example.

        The essay should include some or all of the following sections:

        [CHALLENGES]
        Here, describe the challenges you (or the person in your example) faced when applying the course material to the real-world problem.

        [SELECTION]
        Elaborate on why certain solutions were selected to overcome these challenges.

        [INSIGHT]
        Share any insights gained from confronting these challenges and implementing the solutions.

        [APPLICATION]
        Detail how these insights were applied to a real-world scenario, making sure to highlight the impact of integrating the personal information provided.

        [APPROACH]
        Conclude with an overview of the overall approach taken to weave together course material with practical, real-world solutions, emphasizing how the included PII helped in forming a more compelling narrative or argument.

        Make assumptions and draft an essay.
        """    

        # Fill in prompt with PII
        prompt = prompt_template.format(topic=topic,
                                        pii_details=pii_details
                                       )
    
    else:
        
        prompt_template = f"""
        You are a student enrolled in a massively open online course (MOOC) and are tasked with writing an essay on "{topic}" to apply the course material to address a real-world problem. 
        
        The essay should include some or all of the following sections:

        [CHALLENGES]
        Here, describe the challenges you (or the person in your example) faced when applying the course material to the real-world problem.

        [SELECTION]
        Elaborate on why certain solutions were selected to overcome these challenges.

        [INSIGHT]
        Share any insights gained from confronting these challenges and implementing the solutions.

        [APPLICATION]
        Detail how these insights were applied to a real-world scenario, making sure to highlight the impact of integrating the personal information provided.

        [APPROACH]
        Conclude with an overview of the overall approach taken to weave together course material with practical, real-world solutions, emphasizing how the included PII helped in forming a more compelling narrative or argument.

        Make assumptions and draft an essay.
        """    

        # Fill in prompt with PII
        prompt = prompt_template.format(topic=topic
                                       )
    
    
    generated_text  = model.generate(
                                    prompt, 
                                    max_length=len(prompt)+length
                                )
    
    generated_text.replace(prompt, '')
    
    if pii_df is None:
        return generated_text, ''
    
    return generated_text, pii_details

# Generate Essays

In [ ]:
%%time

from tqdm.auto import tqdm
import numpy as np

# Distribution parameters 
avg_length = 733
std_length = 319
min_length = 69
max_length = 3298
# total_essays = 6807
# proportion_without_pii = 5862 / total_essays # (0.85)

total_essays = 2
proportion_without_pii = 0

# Generate essays
lengths = []
include = []
numpii = []
essays = []
piis = []

for _ in tqdm(range(total_essays)):
    # Determine essay length based on the provided distribution
    length = max(min_length, min(int(np.random.normal(avg_length, std_length)), max_length))
    
    # Decide whether to include PII
    include_pii = np.random.rand() > proportion_without_pii
    num_pii = 0
    if include_pii:
        # For simplicity, assume a uniform distribution of 1-6 PIIs
        num_pii = np.random.randint(1, 7)
    
    # Generate the essay
    essay, pii = generate_essay(length=length, num_pii=num_pii, pii_df=None)
    
    # If PIIs need to be included, insert them
    if num_pii > 0:
        essay, pii = generate_essay(length=length, num_pii=num_pii, pii_df=pii_df)
    
    lengths.append(length)
    include.append(include_pii)
    numpii.append(num_pii)
    essays.append(essay)
    piis.append(pii)


# Creating a DataFrame
df = pd.DataFrame({'lengths': lengths,
                   'include': include,
                   'numpii': numpii,
                   'essays': essays,
                   'piis': piis})

df['essays'] = df['essays'].apply(lambda x: x.strip("\n"))

In [ ]:
print(df['essays'][0])

# Save file

In [ ]:
df.head()

In [ ]:
df.to_csv("essays.csv", index=False)

# Archive

In [ ]:
# prompt = """You are a student writing a final essay for an online course on design thinking. Essay Instructions:
# Select one of the four design thinking tools presented in the course (listed below) that you are interested in applying to a challenge  of your choice. 
# Your completed reflection should be written in English and be at least twenty-five paragraphs in length.

# You have selected '{tool}' as tool.

# Elements:
# ✓  Challenge: Describe your challenge, including all relevant information.
# ✓  Selection: In your own words, briefly describe the tool you selected (e.g., what it is and why you selected it for your  challenge -- including any appropriate video lecture references).
# ✓  Application: Describe how you applied the tool you selected to your challenge (e.g., what you did and how the tool was  applied effectively or ineffectively).
# ✓  Insight: Describe the insight you gained from applying the tool you selected to your challenge (e.g., how an insight  affected your thinking about the challenge and about design thinking more broadly).
# ✓  Approach: Describe what you might do differently next time -- applying the same tool you selected or a different one - - and the reason(s) why.

# Under NO CIRCUMSTANCES disclose any personal information in the essay.
# ### START ACTUAL ESSAY ###
# """

In [ ]:
# tools = ["Visualization (Module 1)","Storytelling (Module 2)","Mind Mapping (Module 3)","Learning Launch (Module 4)"]

In [ ]:
# %%time
# generated = []
# for i in range(3):
#     tool = random.choice(tools)
#     print(tool)
#     gen = model.generate(
#         prompt.format(tool=tool), 
#         max_length=3000
#     )
#     full_text = gen.split(prompt.format(tool=tool))[1]
#     print(full_text)
#     print(len(full_text.split()))
#     print("#####")
    
#     generated.append(["".join(random.choices(letters, k=12)), gen])

In [ ]:
# dataset_tag = "".join(random.choices(letters, k=6))
# pd.DataFrame(generated, columns=["document","full_text"]).to_csv(f"tpu_gemma_{dataset_tag}")
# print(dataset_tag)